In [1]:
import time
import numpy as np
import random
import os
import copy
import math

import pandas as pd
from sklearn import preprocessing
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score
from sklearn.preprocessing import minmax_scale
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier
#from xgboost import XGBClassifier
from scipy.special import softmax
import scipy.stats as ss

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns;
# sns.set_theme(color_codes=True)
import warnings


In [2]:
warnings.filterwarnings("ignore")

random_seed = 0

torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(random_seed)
random.seed(random_seed)

#torch.cuda.set_device("cuda:0")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.environ['CUDA_VISIBLE_DEVICES'] = '1'


In [3]:
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0.0001, path='checkpoint_multi.pt', trace_func=print):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.

            verbose (bool): If True, prints a message for each validation loss improvement.

            delta (float): Minimum change in the monitored quantity to qualify as an improvement.

            path (str): Path for the checkpoint to be saved to.

            trace_func (function): trace print function.

        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_loss, model):

        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter > self.patience:
                self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''Saves model when validation loss decrease.'''
        if self.verbose:
            self.trace_func(
                f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

# 定义用户提供的DNN模型
class DNN(nn.Module):
    def __init__(self, In_Nodes, Modules):
        super(DNN, self).__init__()
        self.Modules = Modules
        self.task_FC1_x = nn.Linear(In_Nodes, Modules, bias=False)
        self.task_FC1_y = nn.Linear(In_Nodes, Modules, bias=False)
        self.task_FC2 = nn.Sequential(nn.Linear(Modules*2, 32), nn.ReLU())
        self.task_FC3 = nn.Sequential(nn.Linear(32, 16), nn.ReLU())
        self.task_FC4 = nn.Linear(16, 5)  # 输出logits

    def forward(self, xg):
        xg_x = self.task_FC1_x(xg)
        xg_y = self.task_FC1_y(xg)
        xg = torch.cat([xg_x.reshape(-1, 1, self.Modules), xg_y.reshape(-1, 1, self.Modules)], dim=1)
        norm = torch.norm(xg, dim=1, keepdim=True)
        xg = xg.div(norm + 1e-8)  # 防止除零
        xg = xg.view(-1, self.Modules * 2)
        xg = self.task_FC2(xg)
        xg = self.task_FC3(xg)
        xg = self.task_FC4(xg)
        return xg

def cal_label(test):
    array = []
    for i in test:
        temp = np.max(i)  # 找到当前行的最大值
        count = 0  # 初始化计数器为0
        for j in i:
            if temp == j: 
                break
            count += 1
        array.append(count)  # 将计数器的值添加到结果列表中
    return np.array(array)  # 将结果列表转换为NumPy数组并返回


In [4]:

# import data
mrna = pd.read_csv(
    'Datasets\mRNA.txt', sep='\t')
meth = pd.read_csv(
    'Datasets\DNA methylation.txt', sep='\t')
mcnv = pd.read_csv(
    'Datasets\CNV.txt', sep='\t')
clincal = pd.read_csv(
    'Datasets\label.txt', sep='\t')

mrna.index = mrna['gene_name']
del mrna['gene_name']

meth.index = meth['gene_name']
del meth['gene_name']

mcnv.index = mcnv['gene_name']
del mcnv['gene_name']

clincal.index = clincal['id']
del clincal['id']

label = clincal['subtype']


mrna=mrna.T
mrna = mrna.iloc[:, :]
meth=meth.T
meth = meth.iloc[:, :]
mcnv=mcnv.T
mcnv=mcnv.iloc[:,:]

mrna = mrna.values
meth = meth.values
mcnv=mcnv.values

min_max_scaler = preprocessing.MinMaxScaler()
mrna = min_max_scaler.fit_transform(mrna)
meth = min_max_scaler.fit_transform(meth)
mcnv = min_max_scaler.fit_transform(mcnv)
label_num, uniques = pd.factorize(label)
label = label_num

# Split data
Xg_train, Xg_test, yg_train, yg_test = train_test_split(mrna, label, test_size=0.2, random_state=42)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(meth, label, test_size=0.2, random_state=42)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(mcnv, label, test_size=0.2, random_state=42)

Xg_train = Xg_train.astype(float)
Xg_test = Xg_test.astype(float)
Xm_train = Xm_train.astype(float)
Xm_test = Xm_test.astype(float)
Xc_train = Xc_train.astype(float)
Xc_test = Xc_test.astype(float)

y_train = np.array(yg_train).flatten().astype(int)
y_test = np.array(yg_test).flatten().astype(int)

Xg = torch.tensor(Xg_train, dtype=torch.float32).to(device)
Xm = torch.tensor(Xm_train, dtype=torch.float32).to(device)
Xc = torch.tensor(Xc_train, dtype=torch.float32).to(device)

Xg_test = torch.tensor(Xg_test, dtype=torch.float32).to(device)
Xm_test = torch.tensor(Xm_test, dtype=torch.float32).to(device)
Xc_test = torch.tensor(Xc_test, dtype=torch.float32).to(device)

y = torch.tensor(y_train, dtype=torch.long).to(device)
y_test = torch.tensor(y_test, dtype=torch.long).to(device)
'''
ds = TensorDataset(Xg, Xm,Xc, y)
loader = DataLoader(ds, batch_size=y_train.shape[0], shuffle=True)

In_Nodes1 = Xg_train.shape[1]
In_Nodes2 = Xm_train.shape[1]
In_Nodes3 = Xc_train.shape[1]'''


'\nds = TensorDataset(Xg, Xm,Xc, y)\nloader = DataLoader(ds, batch_size=y_train.shape[0], shuffle=True)\n\nIn_Nodes1 = Xg_train.shape[1]\nIn_Nodes2 = Xm_train.shape[1]\nIn_Nodes3 = Xc_train.shape[1]'

In [5]:

class BoostedDNNFeatureExtractor:
    def __init__(self, base_model_class, n_estimators=10, lr=0.0001, modules=128, max_epochs=30000):
        self.base_model_class = base_model_class
        self.n_estimators = n_estimators
        self.lr = lr
        self.modules = modules
        self.max_epochs = max_epochs
        self.models = []
        self.weights = []
        
    def _calculate_residual(self, y_true, y_pred_logits):
        """计算伪残差（交叉熵的负梯度）"""
        probs = torch.softmax(y_pred_logits, dim=1)
        one_hot = torch.eye(5)[y_true.long()].to(y_pred_logits.device)
        return one_hot - probs
    
    def fit(self, X, y, in_nodes):
        """训练Boosting模型作为特征提取器"""
        device = X.device if isinstance(X, torch.Tensor) else 'cpu'
        X = torch.as_tensor(X, dtype=torch.float32, device=device)
        y = torch.as_tensor(y, dtype=torch.long, device=device)
        y_onehot = torch.eye(5)[y].float().to(device)
        residual = y_onehot.clone()
        
        for i in range(self.n_estimators):
            # 创建并训练当前弱学习器
            model = self.base_model_class(In_Nodes=in_nodes, Modules=self.modules).to(device)
            optimizer = optim.Adam(model.parameters(), lr=self.lr, weight_decay=1e-4)
            early_stopping = EarlyStopping(patience=20, verbose=False)
            
            # 训练当前模型拟合残差
            dataset = TensorDataset(X, residual)
            loader = DataLoader(dataset, batch_size=64, shuffle=True)
            
            for epoch in range(self.max_epochs):
                model.train()
                total_loss = 0.0
                for batch_X, batch_residual in loader:
                    optimizer.zero_grad()
                    pred_logits = model(batch_X)
                    loss = nn.MSELoss()(pred_logits, batch_residual)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    total_loss += loss.item()
                
                early_stopping(total_loss, model)
                if early_stopping.early_stop:
                    break
            
            # 保存模型并更新残差
            with torch.no_grad():
                model.eval()
                pred_logits = model(X)
                residual_current = self._calculate_residual(y, pred_logits)
                weight = 1.0 / (torch.norm(residual_current).item() + 1e-8)
                self.weights.append(weight)
                self.models.append(model)
                residual = residual_current
    
    def extract_features(self, X, layer_index=-2):
        """
        从Boosting模型中提取特征
        layer_index: 指定从哪一层提取特征，-2表示倒数第二层
        """
        device = next(self.models[0].parameters()).device
        X = torch.as_tensor(X, dtype=torch.float32, device=device)
        
        all_features = []
        
        for model in self.models:
            with torch.no_grad():
                # 获取倒数第二层的输出作为特征
                xg_x = model.task_FC1_x(X)
                xg_y = model.task_FC1_y(X)
                xg = torch.cat([xg_x.reshape(-1, 1, model.Modules), 
                                xg_y.reshape(-1, 1, model.Modules)], dim=1)
                norm = torch.norm(xg, dim=1, keepdim=True)
                xg = xg.div(norm + 1e-8)
                xg = xg.view(-1, model.Modules * 2)
                
                # 通过中间层
                features = model.task_FC2(xg)
                features = model.task_FC3(features)
                
                all_features.append(features)
        
        # 加权融合所有弱学习器的特征
        weighted_features = []
        for features, weight in zip(all_features, self.weights):
            weighted_features.append(features * weight)
        
        # 平均融合
        fused_features = torch.stack(weighted_features).mean(dim=0)
        
        return fused_features.cpu().detach().numpy()


In [ ]:
# 在定义 DNN 和 BoostedDNN 后添加以下代码

class MidFusionAttentionDNN(nn.Module):
    def __init__(self, In_Nodes1, In_Nodes2, In_Nodes3, Modules):
        super(MidFusionAttentionDNN, self).__init__()
        self.Modules = Modules
        
        # 三路输入各自的特征提取
        self.task_FC1_x1 = nn.Linear(In_Nodes1, Modules, bias=False)
        self.task_FC1_y1 = nn.Linear(In_Nodes1, Modules, bias=False)
        
        self.task_FC1_x2 = nn.Linear(In_Nodes2, Modules, bias=False)
        self.task_FC1_y2 = nn.Linear(In_Nodes2, Modules, bias=False)
        
        self.task_FC1_x3 = nn.Linear(In_Nodes3, Modules, bias=False)
        self.task_FC1_y3 = nn.Linear(In_Nodes3, Modules, bias=False)
        
        # 注意力机制
        self.softmax = nn.Softmax(dim=-1)
        
        # 融合后的网络
        self.task_FC2 = nn.Sequential(nn.Linear(Modules * 6, 64), nn.ReLU())
        self.task_FC3 = nn.Sequential(nn.Linear(64, 32), nn.ReLU())
        self.task_FC4 = nn.Linear(32, 5)  # 5类输出

    def forward(self, xg, xm, xc):
        # 每路输入提取双路特征
        xg_x = self.task_FC1_x1(xg)
        xg_y = self.task_FC1_y1(xg)
        
        xm_x = self.task_FC1_x2(xm)
        xm_y = self.task_FC1_y2(xm)
        
        xc_x = self.task_FC1_x3(xc)
        xc_y = self.task_FC1_y3(xc)
        
        # 重构成 (batch, 2, Modules) 用于注意力
        xg_feat = torch.cat([xg_x.reshape(-1, 1, self.Modules), 
                            xg_y.reshape(-1, 1, self.Modules)], dim=1)
        xm_feat = torch.cat([xm_x.reshape(-1, 1, self.Modules), 
                            xm_y.reshape(-1, 1, self.Modules)], dim=1)
        xc_feat = torch.cat([xc_x.reshape(-1, 1, self.Modules), 
                            xc_y.reshape(-1, 1, self.Modules)], dim=1)
        
        # 归一化 - 使用非原地操作
        xg_feat = xg_feat / (torch.norm(xg_feat, dim=1, keepdim=True) + 1e-8)
        xm_feat = xm_feat / (torch.norm(xm_feat, dim=1, keepdim=True) + 1e-8)
        xc_feat = xc_feat / (torch.norm(xc_feat, dim=1, keepdim=True) + 1e-8)
        
        # 简化的融合策略：拼接所有特征
        xg_flat = xg_feat.view(-1, self.Modules * 2)
        xm_flat = xm_feat.view(-1, self.Modules * 2)
        xc_flat = xc_feat.view(-1, self.Modules * 2)
        
        fused = torch.cat([xg_flat, xm_flat, xc_flat], dim=1)
        
        # 后续处理
        x = self.task_FC2(fused)
        x = self.task_FC3(x)
        x = self.task_FC4(x)
        return x

# 修改 BoostedDNN 类以支持中期融合
class BoostedMidFusionDNN:
    def __init__(self, base_model_class, n_estimators=5, lr=0.001, modules=64, max_epochs=100, path=''):
        self.base_model_class = base_model_class
        self.n_estimators = n_estimators
        self.lr = lr
        self.modules = modules
        self.max_epochs = max_epochs
        self.models = []
        self.weights = []
        self.path = path

    def _calculate_residual(self, y_true, y_pred_logits):
        probs = torch.softmax(y_pred_logits, dim=1)
        one_hot = torch.eye(5)[y_true.long()].to(y_pred_logits.device)
        return one_hot - probs

    def fit(self, X_list, y, in_nodes_list):
        """X_list: [Xg, Xm, Xc]"""
        device = X_list[0].device if isinstance(X_list[0], torch.Tensor) else 'cpu'
        
        # 转换为tensor
        X_tensors = []
        for X in X_list:
            if not isinstance(X, torch.Tensor):
                X_tensors.append(torch.as_tensor(X, dtype=torch.float32, device=device))
            else:
                X_tensors.append(X.to(device))
        
        y = torch.as_tensor(y, dtype=torch.long, device=device)
        y_onehot = torch.eye(5)[y].float().to(device)
        residual = y_onehot.clone()
        
        for i in range(self.n_estimators):
            # 创建中期融合模型
            model = self.base_model_class(
                In_Nodes1=in_nodes_list[0],
                In_Nodes2=in_nodes_list[1],
                In_Nodes3=in_nodes_list[2],
                Modules=self.modules
            ).to(device)
            
            optimizer = optim.Adam(model.parameters(), lr=self.lr, weight_decay=1e-4)
            early_stopping = EarlyStopping(patience=20, verbose=False, path=self.path)
            
            # 创建数据集
            dataset = TensorDataset(*X_tensors, residual)
            loader = DataLoader(dataset, batch_size=64, shuffle=True)
            
            for epoch in range(self.max_epochs):
                model.train()
                total_loss = 0.0
                for batch_data in loader:
                    batch_Xg, batch_Xm, batch_Xc, batch_residual = batch_data
                    
                    optimizer.zero_grad()
                    pred_logits = model(batch_Xg, batch_Xm, batch_Xc)
                    loss = nn.MSELoss()(pred_logits, batch_residual)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    total_loss += loss.item()
                
                early_stopping(total_loss, model)
                if early_stopping.early_stop:
                    break
            
            # 计算动态权重
            with torch.no_grad():
                model.eval()
                pred_logits = model(*X_tensors)
                residual_current = self._calculate_residual(y, pred_logits)
                weight = 1.0 / (torch.norm(residual_current).item() + 1e-8)
                self.weights.append(weight)
                self.models.append(model)
                residual = residual_current

    def predict(self, X_list):
        device = next(self.models[0].parameters()).device
        
        # 确保输入是tensor
        X_tensors = []
        for X in X_list:
            if not isinstance(X, torch.Tensor):
                X_tensors.append(torch.as_tensor(X, dtype=torch.float32, device=device))
            else:
                X_tensors.append(X.to(device))
        
        final_logits = torch.zeros(len(X_tensors[0]), 5, device=device)
        
        for model, weight in zip(self.models, self.weights):
            with torch.no_grad():
                logits = model(*X_tensors)
                final_logits += weight * logits
        
        return final_logits

# 使用示例
def train_mid_fusion():
    # 数据准备（与GBdnn相同）
    Xg_train, Xg_test, yg_train, yg_test = train_test_split(mrna, label, test_size=0.2, random_state=42)
    Xm_train, Xm_test, ym_train, ym_test = train_test_split(meth, label, test_size=0.2, random_state=42)
    Xc_train, Xc_test, yc_train, yc_test = train_test_split(mcnv, label, test_size=0.2, random_state=42)
    
    y_train = np.array(yg_train).flatten().astype(int)
    y_test = np.array(yg_test).flatten().astype(int)
    
    # 转换为tensor（可选）
    Xg_train_tensor = torch.tensor(Xg_train, dtype=torch.float32).to(device)
    Xm_train_tensor = torch.tensor(Xm_train, dtype=torch.float32).to(device)
    Xc_train_tensor = torch.tensor(Xc_train, dtype=torch.float32).to(device)
    
    # 初始化并训练中期融合模型
    model = BoostedMidFusionDNN(
        base_model_class=MidFusionAttentionDNN,
        n_estimators=10,
        lr=0.0001,
        modules=128,
        max_epochs=30000,
        path='model/mid_fusion.pt'
    )
    
    # 输入是三模态数据列表
    model.fit(
        X_list=[Xg_train, Xm_train, Xc_train],  # 可以是numpy或tensor
        y=y_train,
        in_nodes_list=[Xg_train.shape[1], Xm_train.shape[1], Xc_train.shape[1]]
    )
    
    # 预测
    y_pred = model.predict([Xg_test, Xm_test, Xc_test])
    accuracy = accuracy_score(y_test, torch.argmax(y_pred, dim=1).cpu().numpy())
    print(f"Mid-Fusion Boosting DNN准确率: {accuracy:.4f}")
    
    return model

# 运行训练
if __name__ == "__main__":
    start = time.time()
    model = train_mid_fusion()
    print(f"time : {time.time() - start:.2f}秒")